# Packages, constants, and useful routines

In [ ]:
import ussa1976
import pymcel as pc
import rebound as rb
import matplotlib.pyplot as plt
import plotly.io as pio
import numpy as np
import pickle
import math
from numba import njit
from scipy.integrate import solve_ivp
import spiceypy as spy

deg = np.pi / 180
rad = 1 / deg

In [ ]:
import spiceypy as spy
import os
import urllib.request

KERNEL_DIR = "kernels"
os.makedirs(KERNEL_DIR, exist_ok=True)

kernels = {
    "naif0012.tls": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls",
    "de430.bsp":    "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de430.bsp",
}

for fname, url in kernels.items():
    path = os.path.join(KERNEL_DIR, fname)
    if not os.path.exists(path):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, path)
    spy.furnsh(path)

print("Kernels Correctly Loaded.")

# General solution to Gauss's orbital equations

We will compare the integration of the system's equations of motion using `rebound` with the direct solution of Gauss's orbital equations.

## Solution using `rebound`

Non-spherical potential model:

In [ ]:
J2 = 0.0010826266835531513
GM_DEFAULT = 398600.4415e9  # m^3/s^2
A_DEFAULT = 6378136.3       # m

# Rebound routine
def aceleracion_no_esferico_J2(simulacion):

  # Retrieve all simulation data
  sim = simulacion.contents

  # Get the particle
  p = sim.particles[1]

  # Position
  x = p.x
  y = p.y
  z = p.z
  r_vec = np.array([x, y, z])
  r = np.linalg.norm(r_vec)

  # Components
  factor = 3 * J2 * GM_DEFAULT * A_DEFAULT**2 / ( 2*r**4 )
  zr2 = z**2 / r**2
  px = factor * (x/r) * (5*zr2 - 1)
  py = factor * (y/r) * (5*zr2 - 1)
  pz = factor * (z/r) * (5*zr2 - 3)

  p.ax += px
  p.ay += py
  p.az += pz

Integration:

In [ ]:
# Initial conditions
a_0 = 8059e3
e_0 = 0.17136
Omega_0 = 45*deg
inc_0 = 28*deg
omega_0 = 30*deg
f_0 = 40*deg
M = GM_DEFAULT / pc.constantes.G

# System creation
sim = rb.Simulation()
sim.units = 'kg', 'm', 's'
sim.add(m=M)
sim.add(m=0, a=a_0, e=e_0, inc=inc_0, Omega=Omega_0, omega=omega_0, f=f_0)

sim.additional_forces = aceleracion_no_esferico_J2
sim.force_is_velocity_dependent = True

# Simulation
deltat = 48*3600
Nt = 1000
ts = np.linspace(sim.t, sim.t + deltat, Nt)
Es = np.zeros((Nt, 5))
Xs = np.zeros((Nt, 6))

for i, t_step in enumerate(ts):
    sim.integrate(t_step)
    p = sim.particles[1]
    o = sim.orbits()[0]
    Es[i] = [o.a, o.e, o.inc, o.Omega, o.omega]
    Xs[i] = [p.x, p.y, p.z, p.vx, p.vy, p.vz]

# Plotting
pio.renderers.default = "png"
fig, axes = plt.subplots(5, 1, sharex=True, figsize=(10, 8))

# Difference in a
axes[0].plot(ts/3600, Es[:, 0] - a_0)
axes[0].set_ylabel(r'$\Delta a$ (m)')
axes[0].grid(True)

# Difference in e
axes[1].plot(ts/3600, Es[:, 1] - e_0)
axes[1].set_ylabel(r'$\Delta e$')
axes[1].grid(True)

# Difference in inclination in degrees
axes[2].plot(ts/3600, (Es[:, 2] - inc_0) * rad)
axes[2].set_ylabel(r'$\Delta i$ (deg)')
axes[2].grid(True)

# Difference in Omega in degrees
axes[3].plot(ts/3600, (Es[:, 3] - Omega_0) * rad)
axes[3].set_ylabel(r'$\Delta \Omega$ (deg)')
axes[3].grid(True)

# Difference in omega in degrees
axes[4].plot(ts/3600, (Es[:, 4] - omega_0) * rad)
axes[4].set_ylabel(r'$\Delta \omega$ (deg)')
axes[4].set_xlabel('Time (s)')
axes[4].grid(True)

plt.tight_layout()
plt.show()

## Perturbations calculated with Gauss's equations

Gauss's orbital equations are:

\begin{aligned}
\frac{\mathrm{d} h}{\mathrm{~d} t} & =r p_s \\
\frac{\mathrm{~d} e}{\mathrm{~d} t} & =\frac{h}{\mu} \sin \theta p_r+\frac{1}{\mu h}\left[\left(h^2+\mu r\right) \cos \theta+\mu e r\right] p_s \\
\frac{\mathrm{~d} \theta}{\mathrm{~d} t} & =\frac{h}{r^2}+\frac{1}{e h}\left[\frac{h^2}{\mu} \cos \theta p_r-\left(r+\frac{h^2}{\mu}\right) \sin \theta p_s\right] \\
\frac{\mathrm{d} \Omega}{\mathrm{~d} t} & =\frac{r}{h \sin i} \sin (\omega+\theta) p_w \\
\frac{\mathrm{~d} i}{\mathrm{~d} t} & =\frac{r}{h} \cos (\omega+\theta) p_w \\
\frac{\mathrm{~d} \omega}{\mathrm{~d} t} & =-\frac{1}{e h}\left[\frac{h^2}{\mu} \cos \theta p_r-\left(r+\frac{h^2}{\mu}\right) \sin \theta p_s\right]-\frac{r \sin (\omega+\theta)}{h \tan i} p_w
\end{aligned}

They are implemented as follows:

In [ ]:
def gauss_ecuaciones_orbitales(t, Y, mu, p_func, p_params):
  h, e, theta, Omega, I, omega = Y

  # Derived orbital quantities
  p = h**2 / mu # semilatus rectum
  r = p / (1 + e * np.cos(theta)) # Distance
  u = omega + theta # Argument of latitude
  q = p / (1 + e) # Perifocal distance
  E = 2*np.arctan(np.sqrt((1-e)/(1+e))*np.tan(theta/2)) # Eccentric anomaly
  M = E - e * np.sin(E) # Mean anomaly

  # State vector using `conics` from SPICE
  X = spy.conics([q, e, I, Omega, omega, M, 0, mu], 0)

  # Calculate the perturbation vector
  p_vec = p_func(t, Y, X, p_params)

  # Find the perturbation vector components in the LVLH frame
  Mxyz2rsw = spy.eul2m(u, I, Omega, 3, 1, 3)
  pr, ps, pw = Mxyz2rsw @ p_vec

  # Gauss's equations
  dh_dt = r * ps

  de_dt = (h / mu) * np.sin(theta) * pr + \
          (1 / (mu * h)) * ((h**2 + mu * r) * np.cos(theta) + mu * e * r) * ps

  dtheta_dt = h / r**2 + (1 / (e * h)) * ((h**2 / mu) * np.cos(theta) * pr - \
                                          (r + h**2 / mu) * np.sin(theta) * ps)

  dOmega_dt = (r / (h * np.sin(I))) * np.sin(omega + theta) * pw

  dI_dt = (r / h) * np.cos(omega + theta) * pw

  domega_dt = -(1 / (e * h)) * ((h**2 / mu) * np.cos(theta) * pr - \
                                (r + h**2 / mu) * np.sin(theta) * ps) - \
              (r * np.sin(omega + theta)) / (h * np.tan(I)) * pw

  return [dh_dt, de_dt, dtheta_dt, dOmega_dt, dI_dt, domega_dt]

Let's test it for the calculation of the orbital perturbations produced by Earth's non-spherical shape. In this case, we can write the function that gives us the perturbing acceleration as follows:

In [ ]:
def p_no_esferico_J2(t, Y, X, p_params):

  # Input parameters
  x, y, z, vx, vy, vz = X
  r = np.linalg.norm([x, y, z])

  J2 = p_params['J2']
  GM = p_params['GM']
  A = p_params['A']

  # Acceleration components
  factor = 3 * J2 * GM * A**2 / ( 2*r**4 )
  zr2 = z**2 / r**2
  px = factor * (x/r) * (5*zr2 - 1)
  py = factor * (y/r) * (5*zr2 - 1)
  pz = factor * (z/r) * (5*zr2 - 3)

  return np.array([px, py, pz])

Let's define the initial conditions:

In [ ]:
# Initial conditions
a_0 = 8059e3
e_0 = 0.17136
Omega_0 = 45*deg
I_0 = 28*deg
omega_0 = 30*deg
theta_0 = 40*deg
mu = GM_DEFAULT

# Derivatives
p_0 = a_0 * (1 - e_0**2)
h_0 = np.sqrt(p_0 * mu)

# Initial conditions
Y0 = [h_0, e_0, theta_0, Omega_0, I_0, omega_0]
p_params = dict(J2=J2, GM=GM_DEFAULT, A=A_DEFAULT)

# Times
ts = np.linspace(0, 48*3600, 1000)

Let's solve and extract the solution:

In [ ]:
# Solve
sol = solve_ivp(gauss_ecuaciones_orbitales, [0, ts[-1]], Y0, t_eval=ts,
                args=(mu, p_no_esferico_J2, p_params), method='Radau', rtol=1e-6)

# Extract all elements
Egauss = sol.y.T

# Extract the elements one by one
hs = sol.y[0]
es = sol.y[1]
thetas = sol.y[2]
Omegas = sol.y[3]
Is = sol.y[4]
omegas = sol.y[5]
ps = hs**2 / mu
aes = ps / (1 - es**2)

Now let's plot a comparison with the solution using `rebound`:

In [ ]:
# Plotting
pio.renderers.default = "png"
fig, axes = plt.subplots(5, 1, sharex=True, figsize=(10, 8))

# Difference in a
axes[0].plot(ts/3600, aes - a_0)
axes[0].plot(ts/3600, Es[:, 0] - a_0, 'r--')
axes[0].set_ylabel(r'$\Delta a$ (m)')
axes[0].grid(True)

# Difference in e
axes[1].plot(ts/3600, es - e_0)
axes[1].plot(ts/3600, Es[:, 1] - e_0, 'r--')
axes[1].set_ylabel(r'$\Delta e$')
axes[1].grid(True)

# Difference in inclination in degrees
axes[2].plot(ts/3600, (Is - inc_0) * rad)
axes[2].plot(ts/3600, (Es[:, 2] - inc_0) * rad, 'r--')
axes[2].set_ylabel(r'$\Delta i$ (deg)')
axes[2].grid(True)

# Difference in Omega in degrees
axes[3].plot(ts/3600, (Omegas - Omega_0) * rad)
axes[3].plot(ts/3600, (Es[:, 3] - Omega_0) * rad, 'r--')
axes[3].set_ylabel(r'$\Delta \Omega$ (deg)')
axes[3].grid(True)

# Difference in omega in degrees
axes[4].plot(ts/3600, (omegas - omega_0) * rad)
axes[4].plot(ts/3600, (Es[:, 4] - omega_0) * rad, 'r--')
axes[4].set_ylabel(r'$\Delta \omega$ (deg)')
axes[4].set_xlabel('Time (s)')
axes[4].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
(Egauss[:, 1] - Es[:, 1]) / abs(Egauss[:, 1] + Es[:, 1] )

We can compare the positions calculated with both methods. To do this, let's write a routine that finds the state vectors from the orbital elements:

In [ ]:
def elementos_a_estados(ts, Ys, mu):
  Xs = np.zeros((len(ts), 6))
  for i, t in enumerate(ts):
    h, e, theta, Omega, I, omega = Ys[i]

    # Derived orbital quantities
    p = h**2 / mu # semilatus rectum
    q = p / (1 + e) # Perifocal distance
    E = 2*np.arctan(np.sqrt((1-e)/(1+e))*np.tan(theta/2)) # Eccentric anomaly
    M = E - e * np.sin(E) # Mean anomaly

    # State vector using `conics` from SPICE
    Xs[i] = spy.conics([q, e, I, Omega, omega, M, 0, mu], 0)
  return Xs

The final position calculated from the solution to Gauss's orbital equations:

In [ ]:
Xgauss = elementos_a_estados(ts, Egauss, mu)

The difference in terms of state vectors is:

In [ ]:
Xs[-1] - Xgauss[-1]

We observe that despite the apparent agreement, the calculation of positions and velocities can have errors of up to several tens of km and several tens of m/s. This means that Gauss's equations are useful for predicting the long-term behavior of the orbit, but not as accurate for predicting the precise position of satellites.

# Solar radiation

Basic parameters:

In [ ]:
Tsol = 5778 # K, effective temperature of the Sun
S0 = pc.constantes.sigma_sb * Tsol**4
R0 = pc.constantes.R_sun

def S_solar(r):
  return S0 * (R0 / r)**2

At Earth's distance:

In [ ]:
S_solar(pc.constantes.au)

Radiation pressure at Earth's mean distance:

In [ ]:
S0 / 1e6

In [ ]:
S_solar(pc.constantes.au) / pc.constantes.c

### Integrating the effect

In [ ]:
# Constants
R = A_DEFAULT
# Initial date
jd0 = 2438400.5 # days
# Convert from ecliptic to J2000
Mecl2equ = spy.pxform('ECLIPJ2000', 'J2000', 0)

def p_radiacion_solar(t, Y, X, p_params):

  # Input parameters
  x, y, z, vx, vy, vz = X
  r_sat = X[:3]

  # Sun's position
  jd = jd0 + t/86400
  et = spy.unitim(jd, 'JDTDB', 'ET') 
  estado, lt = spy.spkezr('10', et, 'J2000', 'NONE', '399')
  r_sol = estado[:3] * 1000  
  r_sol_equ = spy.mxv(Mecl2equ, r_sol)
  d_sol = np.linalg.norm(r_sol_equ)
  u_hat = r_sol_equ / d_sol

  # Determine whether the satellite is illuminated by the Sun
  teta = np.arccos((r_sol_equ @ r_sat) / (np.linalg.norm(r_sol_equ) * np.linalg.norm(r_sat)))
  teta1 = np.arccos(R/np.linalg.norm(r_sat))
  teta2 = np.arccos(R/np.linalg.norm(r_sol_equ))
  nu = 1 if (teta1 + teta2 - teta)>0 else 0

  # Calculate the acceleration due to radiation pressure
  psr = -nu * (S_solar(d_sol)/pc.constantes.c) * p_params['CR'] * p_params['Ascm'] * u_hat

  return psr

Initial conditions for Example 10.9

Angular momentum: $h_0=63,383.4 \mathrm{~km}^2 / \mathrm{s}$

Eccentricity: $e_0=0.025422$

Right ascension of the node: $\Omega_0=45.3812^{\circ}$

Inclination: $i_0=88.3924^{\circ}$

Argument of perigee: $\omega_0=227.493^{\circ}$

True anomaly: $\theta_0=343.427^{\circ}$

In [ ]:
# Initial conditions
h_0 = 63383.4 * 1e6 # Convert km^2/s to m^2/s
e_0 = 0.025422
Omega_0 = 45.3812 * deg # Convert degrees to radians
I_0 = 88.3924 * deg   # Convert degrees to radians
omega_0 = 227.493 * deg # Convert degrees to radians
theta_0 = 343.427 * deg # Convert degrees to radians
mu = GM_DEFAULT
Y0 = [h_0, e_0, theta_0, Omega_0, I_0, omega_0]

# Parameters
p_params = dict(CR=2, Ascm=2)

Let's test the routine:

In [ ]:
# Derived orbital quantities using the initial conditions
p_0 = h_0**2 / mu # semilatus rectum
r_0 = p_0 / (1 + e_0 * np.cos(theta_0)) # Distance
u_0 = omega_0 + theta_0 # Argument of latitude
q_0 = p_0 / (1 + e_0) # Perifocal distance
E_0 = 2*np.arctan(np.sqrt((1-e_0)/(1+e_0))*np.tan(theta_0/2)) # Eccentric anomaly
M_0 = E_0 - e_0 * np.sin(E_0) # Mean anomaly
a_0 = p_0 / (1 - e_0**2)

# State vector using `conics` from SPICE for the initial conditions
X_0 = spy.conics([q_0, e_0, I_0, Omega_0, omega_0, M_0, 0, mu], 0)

# Test the p_radiacion_solar routine with the initial conditions
# We need an initial time t_0. We will use 0 for testing.
t_0 = 0
X_0, p_radiacion_solar(t_0, Y0, X_0, p_params)

Let's integrate:

In [ ]:
ts = np.linspace(0, 1*86400, 100)

# Solve
sol = solve_ivp(gauss_ecuaciones_orbitales, [0, ts[-1]], Y0, t_eval=ts,
                args=(mu, p_radiacion_solar, p_params), method='Radau', rtol=1e-6)

# Extract all elements
Egauss = sol.y.T

# Extract the elements one by one
hs = sol.y[0]
es = sol.y[1]
thetas = sol.y[2]
Omegas = sol.y[3]
Is = sol.y[4]
omegas = sol.y[5]
ps = hs**2 / mu
aes = ps / (1 - es**2)

Now let's plot the result:

In [ ]:
# Plotting
pio.renderers.default = "png"
fig, axes = plt.subplots(6, 1, sharex=True, figsize=(12, 8))

# Difference in a
tfac = 86400
axes[0].plot(ts/tfac, (hs - h_0)/1e6)
axes[0].set_ylabel(r'$\Delta h$ [km$^2$/s]')
axes[0].grid(True)

# Difference in e
axes[1].plot(ts/tfac, es - e_0)
axes[1].set_ylabel(r'$\Delta e$')
axes[1].grid(True)

# Difference in inclination in degrees
axes[2].plot(ts/tfac, (Is - I_0) * rad)
axes[2].set_ylabel(r'$\Delta i$ (deg)')
axes[2].grid(True)

# Difference in Omega in degrees
axes[3].plot(ts/tfac, (Omegas - Omega_0) * rad)
axes[3].set_ylabel(r'$\Delta \Omega$ (deg)')
axes[3].grid(True)

# Difference in omega in degrees
axes[4].plot(ts/tfac, (omegas - omega_0) * rad)
axes[4].set_ylabel(r'$\Delta \omega$ (deg)')
axes[4].set_xlabel('Time (s)')
axes[4].grid(True)

axes[5].plot(ts/tfac, (aes - a_0)/1e3)
axes[5].set_ylabel(r'$\Delta a$ [km]')
axes[5].grid(True)

plt.tight_layout()
plt.show()

## Exercises

### Exercise 1

In [ ]:
zs = np.linspace(0,1000,1000)
data = ussa1976.compute(zs*1e3, variables=['rho'])

In [ ]:
rho = np.array(data.rho)

In [ ]:
mu = GM_DEFAULT
rE = A_DEFAULT
mu, rE

In [ ]:
CR = 1
CD = 2
c = pc.constantes.c
zs = np.linspace(1, 1000, 1000)
S = S0 * (R0 / (rE + zs))**2
v_2 = (mu) / (rE + zs)
p_sr_atm = (2 * S * CR) / (c * rho *1e9 * v_2 * CD)

In [ ]:
pio.renderers.default = "png"
plt.figure(figsize=(8, 5))
plt.plot(zs, p_sr_atm)
plt.yscale('log')
plt.xlabel('Altitude (km)')
plt.ylabel('P_sr / P_atm ')
plt.grid(True)
plt.show()

## Exercise 10.11

In [ ]:
# Initial conditions
h_0 = 129640 * 1e6 # Convert km^2/s to m^2/s
e_0 = 0.0001
Omega_0 = 0 * deg # Convert degrees to radians
I_0 = 1 * deg   # Convert degrees to radians
omega_0 = 0 * deg # Convert degrees to radians
theta_0 = 0 * deg # Convert degrees to radians
#a_0 = 42164 * 1e3
#T_0 = 23.9343 * 3600 # In seconds
mu = GM_DEFAULT
Y0 = [h_0, e_0, theta_0, Omega_0, I_0, omega_0]

# Parameters
p_params = dict(CR=2, Ascm=2)

In [ ]:
mu

In [ ]:
pc.constantes.mu_moon

In [ ]:
mu_luna = pc.constantes.mu_moon

In [ ]:
jd0 =  2454283.0

In [ ]:
def p_gravitacional_lunar(t, Y, X, p_params):

  # Input parameters
  x, y, z, vx, vy, vz = X
  r_sat = X[:3]

  # Moon's position relative to Earth
  jd = jd0 + t/86400
  et = spy.unitim(jd, 'JDTDB', 'ET')
  estado, lt = spy.spkezr('301', et, 'J2000', 'NONE', '399')
  r_luna = estado[:3] * 1000  
  r_luna_equ = spy.mxv(Mecl2equ, r_luna)

  # r_p magnitude
  r_luna_mag = np.linalg.norm(r_luna_equ)

  # r_ps
  r_ps = r_luna_equ - r_sat
  r_ps_mag = np.linalg.norm(r_ps)

  # Calculate q
  q = (r_sat @ ((2 * r_luna_equ) - r_sat)) / (r_luna_mag**2)

  # q function
  f_q = q * (((q**2) - (3 * q) + 3)/(1 + (1 - q)**1.5))
  print(f_q)
  print(q)
  # Perturbation vector
  p_vec = (mu_luna/r_ps_mag**3) * (f_q * r_luna_equ - r_sat)
  print(p_vec)

  return p_vec

In [ ]:
# Derived orbital quantities using the initial conditions
p_0 = h_0**2 / mu # semilatus rectum
r_0 = p_0 / (1 + e_0 * np.cos(theta_0)) # Distance
u_0 = omega_0 + theta_0 # Argument of latitude
q_0 = p_0 / (1 + e_0) # Perifocal distance
E_0 = 2*np.arctan(np.sqrt((1-e_0)/(1+e_0))*np.tan(theta_0/2)) # Eccentric anomaly
M_0 = E_0 - e_0 * np.sin(E_0) # Mean anomaly
a_0 = p_0 / (1 - e_0**2)

# State vector using `conics` from SPICE for the initial conditions
X_0 = spy.conics([q_0, e_0, I_0, Omega_0, omega_0, M_0, 0, mu], 0)

# Test the p_radiacion_solar routine with the initial conditions
# We need an initial time t_0. We will use 0 for testing.
t_0 = 0
X_0, p_gravitacional_lunar(t_0, Y0, X_0, p_params)

In [ ]:
ts = np.linspace(0, 60*86400, 1000)

# Solve
sol = solve_ivp(gauss_ecuaciones_orbitales, [0, ts[-1]], Y0, t_eval=ts,
                args=(mu, p_gravitacional_lunar, p_params), method='Radau', rtol=1e-6)

# Extract all elements
Egauss = sol.y.T

# Extract the elements one by one
hs = sol.y[0]
es = sol.y[1]
thetas = sol.y[2]
Omegas = sol.y[3]
Is = sol.y[4]
omegas = sol.y[5]
ps = hs**2 / mu
aes = ps / (1 - es**2)

In [ ]:
# Plotting
pio.renderers.default = "png"
fig, axes = plt.subplots(6, 1, sharex=True, figsize=(12, 8))

tfac = 86400

# Difference in a
axes[0].plot(ts/tfac, (hs - h_0)/1e6)
axes[0].set_ylabel(r'$\Delta h$ [km$^2$/s]')
axes[0].grid(True)

# Difference in e
axes[1].plot(ts/tfac, es - e_0)
axes[1].set_ylabel(r'$\Delta e$')
axes[1].grid(True)

# Difference in inclination in degrees
axes[2].plot(ts/tfac, (Is - I_0) * rad)
axes[2].set_ylabel(r'$\Delta i$ (deg)')
axes[2].grid(True)

# Difference in Omega in degrees
axes[3].plot(ts/tfac, (Omegas - Omega_0) * rad)
axes[3].set_ylabel(r'$\Delta \Omega$ (deg)')
axes[3].grid(True)

# Difference in omega in degrees
axes[4].plot(ts/tfac, (omegas - omega_0) * rad)
axes[4].set_ylabel(r'$\Delta \omega$ (deg)')
axes[4].set_xlabel('Time (s)')
axes[4].grid(True)

axes[5].plot(ts/tfac, (aes - a_0)/1e3)
axes[5].set_ylabel(r'$\Delta a$ [km]')
axes[5].grid(True)

plt.tight_layout()
plt.show()

### Exercise 10.12

In [ ]:
mu_sol = pc.constantes.mu_sun

In [ ]:
mu_sol

In [ ]:
def p_gravitacional_solar(t, Y, X, p_params):

  # Input parameters
  x, y, z, vx, vy, vz = X
  r_sat = X[:3]

  # Sun's position relative to Earth
  jd = jd0 + t/86400
  et = spy.unitim(jd, 'JDTDB', 'ET')  
  estado, lt = spy.spkezr('10', et, 'J2000', 'NONE', '399')
  r_sol = estado[:3] * 1000  
  r_sol_equ = spy.mxv(Mecl2equ, r_sol)

  # r_p magnitude
  r_sol_mag = np.linalg.norm(r_sol_equ)

  # r_ps
  r_ps = r_sol_equ - r_sat
  r_ps_mag = np.linalg.norm(r_ps)

  # Calculate q
  q = (r_sat @ ((2 * r_sol_equ) - r_sat)) / (r_sol_mag**2)

  # q function
  f_q = q * (((q**2) - (3 * q) + 3)/(1 + (1 - q)**1.5))
  print(f_q)
  print(q)
  # Perturbation vector
  p_vec = (mu_sol/r_ps_mag**3) * (f_q * r_sol_equ - r_sat)
  print(p_vec)

  return p_vec

In [ ]:
# Derived orbital quantities using the initial conditions
p_0 = h_0**2 / mu # semilatus rectum
r_0 = p_0 / (1 + e_0 * np.cos(theta_0)) # Distance
u_0 = omega_0 + theta_0 # Argument of latitude
q_0 = p_0 / (1 + e_0) # Perifocal distance
E_0 = 2*np.arctan(np.sqrt((1-e_0)/(1+e_0))*np.tan(theta_0/2)) # Eccentric anomaly
M_0 = E_0 - e_0 * np.sin(E_0) # Mean anomaly
a_0 = p_0 / (1 - e_0**2)

# State vector using `conics` from SPICE for the initial conditions
X_0 = spy.conics([q_0, e_0, I_0, Omega_0, omega_0, M_0, 0, mu], 0)

# Test the p_radiacion_solar routine with the initial conditions
# We need an initial time t_0. We will use 0 for testing.
t_0 = 0
X_0, p_gravitacional_solar(t_0, Y0, X_0, p_params)

In [ ]:
ts = np.linspace(0, 750*86400, 1000)

# Solve
sol = solve_ivp(gauss_ecuaciones_orbitales, [0, ts[-1]], Y0, t_eval=ts,
                args=(mu, p_gravitacional_solar, p_params), method='Radau', rtol=1e-6)

# Extract all elements
Egauss = sol.y.T

# Extract the elements one by one
hs = sol.y[0]
es = sol.y[1]
thetas = sol.y[2]
Omegas = sol.y[3]
Is = sol.y[4]
omegas = sol.y[5]
ps = hs**2 / mu
aes = ps / (1 - es**2)

In [ ]:
# Plotting
pio.renderers.default = "png"
fig, axes = plt.subplots(6, 1, sharex=True, figsize=(12, 8))

tfac = 86400
def suavizar(y, ventana=15):
  return np.convolve(y, np.ones(ventana)/ ventana, mode='same')

# Difference in a
axes[0].plot(ts/tfac, (hs - h_0)/1e6)
axes[0].plot(ts/tfac, suavizar((hs - h_0)/1e6))
axes[0].set_ylabel(r'$\Delta h$ [km$^2$/s]')
axes[0].grid(True)

# Difference in e
axes[1].plot(ts/tfac, es - e_0)
axes[1].plot(ts/tfac, suavizar(es - e_0))
axes[1].set_ylabel(r'$\Delta e$')
axes[1].grid(True)

# Difference in inclination in degrees
axes[2].plot(ts/tfac, (Is - I_0) * rad)
axes[2].set_ylabel(r'$\Delta i$ (deg)')
axes[2].grid(True)

# Difference in Omega in degrees
axes[3].plot(ts/tfac, (Omegas - Omega_0) * rad)
axes[3].set_ylabel(r'$\Delta \Omega$ (deg)')
axes[3].grid(True)

# Difference in omega in degrees
axes[4].plot(ts/tfac, (omegas - omega_0) * rad)
axes[4].plot(ts/tfac, suavizar((omegas - omega_0) * rad))
axes[4].set_ylabel(r'$\Delta \omega$ (deg)')
axes[4].set_xlabel('Time (s)')
axes[4].grid(True)

axes[5].plot(ts/tfac, (aes - a_0)/1e3)
axes[5].plot(ts/tfac, suavizar((aes - a_0)/1e3))
axes[5].set_ylabel(r'$\Delta a$ [km]')
axes[5].grid(True)

plt.tight_layout()
plt.show()